In [ ]:
%load_ext autoreload
%autoreload 2
%reset -f

# Imports

In [ ]:
from locallib.picarrodb import *
from locallib.query import *
from locallib.box import *
from locallib.query import *

import pandas as pd
import geopandas as gpd
import contextily as ctx
import pandas as pd
import os

import sys
import os

%matplotlib widget
import matplotlib.pyplot as plt
from shapely import wkt
from shapely.ops import unary_union
import matplotlib.pyplot as plt
from h3 import *
from shapely.geometry import Point
from shapely.geometry import LineString, Point
import numpy as np

os.chdir('/home/sandbox/personal-repos/DA-3507/dump')
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../lib')))
from custom_pandas import *

# Configuration

In [ ]:
street_length = 10
sampling_period = 5
line_length = 10

## Query the surveys

In [ ]:
a = get_reports('Cadent',years = [2026]).execute([EU2_Conn])
report_bc = a.iloc[[800]].copy()

## Query the geometries

In [ ]:
report_bc.db.set_query(query_SurveyH3Aggregation_byReport(report_table = '#TempReport'))
agg_segments =report_bc.db.execute([EU2_Conn], source_col = 'ReportId', temp_table_name = '#TempReport')
report_bc.db.set_query(query_Segments_byReport(report_table = '#TempReport'))
segments = report_bc.db.execute([EU2_Conn], source_col = 'ReportId', temp_table_name = '#TempReport')

## Isolate a single survey

In [ ]:
l = 0
surveys = segments['SurveyId'].unique()
#survey = segments[segments['SurveyId'] == surveys[l]]
survey = segments
survey['Breadcrumb_wkt'] = survey['Breadcrumb'].apply(wkt.loads)
survey_gdf = gpd.GeoDataFrame(survey, geometry = 'Breadcrumb_wkt', crs = 'EPSG:4326')
utm_crs = survey_gdf.estimate_utm_crs()
survey_gdf = survey_gdf.to_crs(utm_crs)

## Prepare the countour geometry

In [ ]:
contour = agg_segments[agg_segments['SurveyId'] == surveys[l]]
contour['Breadcrumb_wkt'] = contour['Breadcrumb'].apply(wkt.loads)
contour_gdf = gpd.GeoDataFrame(contour, geometry = 'Breadcrumb_wkt', crs = 'EPSG:4326')
contour_gdf = contour_gdf.to_crs(utm_crs)
contour_gdf['offset'] = contour_gdf.geometry.buffer(street_length/2, cap_style=2)


## Create a evenly spaced grid over the line

In [ ]:
def sample_linearly(line, distance):
    if not isinstance(line, LineString):
        return []
    length = line.length
    # Compute evenly spaced distances along the line, but exclude endpoints
    num_points = int(np.floor(length / distance)) + 1
    if np.isclose(num_points * distance, length):
        distances = np.linspace(0, length, num=num_points + 1)
    else:
        distances = np.arange(0, length, distance)
        if length not in distances:
            distances = np.append(distances, length)
    # Remove starting (0) and ending (length) point
    distances = [d for d in distances if not np.isclose(d, 0) and not np.isclose(d, length)]
    return [line.interpolate(d) for d in distances]

survey_gdf['EvenlySpaced'] = survey_gdf.geometry.apply(lambda line: sample_linearly(line, sampling_period))
#survey_gdf['EvenlySpaced'] = survey_gdf.geometry.segmentize(sampling_period)

# Create the geometry from list of points, plot the result
survey_gdf['EvenlySpacedLine'] = survey_gdf['EvenlySpaced'].apply(lambda pts: LineString(pts) if len(pts) >= 2 else None)
#survey_gdf['EvenlySpacedLine'] = survey_gdf['EvenlySpaced']

# Create a line perpendicular line at each vertex

In [ ]:
import numpy as np
from shapely.geometry import LineString, Point

def unit_vector(vector):
    """ Returns the unit vector of the vector.  """
    return vector / np.linalg.norm(vector)

def perpendicular_vector(dx, dy):
    """ Returns a perpendicular vector (to the left of direction of travel). """
    return np.array([-dy, dx])

def perp_lines_at_vertices(line, length=10):
    """Generate perpendicular lines at each vertex of a LineString."""
    coords = list(line.coords)
    perps = []
    n = len(coords)

    for i, (x, y) in enumerate(coords):
        # For endpoints, use segment direction from second/previous point
        if n < 2:
            continue  # Not enough points
        elif i == 0:
            x2, y2 = coords[i + 1]
            dx, dy = x2 - x, y2 - y
        elif i == n - 1:
            x1, y1 = coords[i - 1]
            dx, dy = x - x1, y - y1
        else:
            # Average both directions for a smoother result
            x1, y1 = coords[i - 1]
            x2, y2 = coords[i + 1]
            dx1, dy1 = x - x1, y - y1
            dx2, dy2 = x2 - x, y2 - y
            dx, dy = unit_vector(np.array([dx1, dy1])) + unit_vector(np.array([dx2, dy2]))
            if np.linalg.norm([dx, dy]) == 0:  # colinear back-to-back turn, fallback
                dx, dy = -dy1, dx1

        direction = unit_vector(np.array([dx, dy]))
        perp = perpendicular_vector(*direction)
        midpoint = np.array([x, y])
        half = (length / 2.0) * perp
        line_pts = [midpoint - half, midpoint + half]
        perps.append(LineString(line_pts))

    return perps

perp_lines = []
for geom in survey_gdf.EvenlySpacedLine:
    if isinstance(geom, LineString):
        perp_lines.extend(perp_lines_at_vertices(geom, length=line_length))

perp_gdf = gpd.GeoDataFrame(geometry=perp_lines, crs=survey_gdf.crs if hasattr(survey_gdf, 'crs') else None)
perp_gdf = perp_gdf.reset_index(drop=True)
perp_gdf.index = pd.Index(range(len(perp_gdf)), name='perp_id')

# Create the intersection for getting the number of passes

In [ ]:
points = gpd.overlay(perp_gdf, survey_gdf, how='intersection', keep_geom_type=False)[['geometry','Order']]
# Expand any MultiPoint into two points (or more, as many as present), else leave as is
expanded_points = []
for geom in points['geometry']:
    if geom.geom_type == 'MultiPoint':
        expanded_points.extend(list(geom.geoms)[:2])  # take first 2 points
    else:
        expanded_points.append(geom)
expanded_points = gpd.GeoDataFrame(geometry=expanded_points, crs=points.crs if hasattr(points, 'crs') else None)
buffered_points = expanded_points.buffer(0.001)

In [ ]:
buffered_points_gdf = gpd.GeoDataFrame(geometry=buffered_points, crs=expanded_points.crs if hasattr(expanded_points, 'crs') else None)
buffered_points_gdf

In [ ]:
joined = gpd.sjoin(buffered_points_gdf, perp_gdf, how='left', predicate='intersects')

In [ ]:
# Join the count of points (number of passes) with perp_gdf by perp_id
count_by_perp = joined['index_right'].value_counts().rename_axis('perp_id').reset_index(name='count')
perp_gdf_with_count = perp_gdf.reset_index().merge(count_by_perp, on='perp_id', how='left').set_index('perp_id')
perp_gdf_with_count

# Check the histograms

In [ ]:
perp_gdf_with_count['count'].value_counts()

In [ ]:

fig,axr = plt.subplots(figsize=(10,10))
perp_gdf_with_count['count'].hist(ax=axr)


# Plotting

In [ ]:
import contextily as ctx

fig, axr = plt.subplots(figsize=(10,10))

survey_gdf.to_crs(epsg=3857).plot(ax=axr, color='blue', alpha=0.5)
perp_gdf_with_count.to_crs(epsg=3857).plot(ax=axr, legend=True, column='count', cmap='Set1')
contour_gdf.offset.to_crs(epsg=3857).plot(ax=axr, color='red', alpha=0.5)

ctx.add_basemap(axr, source=ctx.providers.OpenStreetMap.Mapnik, crs='EPSG:3857')

In [ ]:
perp_gdf_with_count['centroid'] = perp_gdf_with_count['geometry'].centroid

In [ ]:
perp_2 = perp_gdf_with_count[perp_gdf_with_count['count'] == 2]
max_dist = 10
line_list = []
line_pts = []
done_line = True
done_list = True
i = 0
# ignoring all warnings as requested
out = []
from scipy.spatial import cKDTree
import numpy as np

# Get centroid coordinates as array for KDTree
centroids = np.array(list(perp_2['centroid'].apply(lambda x: (x.x, x.y))))
tree = cKDTree(centroids)
used = np.zeros(len(perp_2), dtype=bool)

for i in range(len(perp_2)):
    if used[i]:
        continue
    # Query KDTree for neighbors within max_dist (including self)
    idxs = tree.query_ball_point(centroids[i], r=max_dist)
    idxs = [idx for idx in idxs if not used[idx]]
    if len(idxs) >= 2:
        # Build line from the found cluster
        line = LineString([perp_2.iloc[idx]['centroid'].coords[0] for idx in idxs])
        line_list.append(line)
        # Mark these as used
        used[idxs] = True
    else:
        # Not enough close points (including itself)
        out = [perp_2.iloc[idx] for idx in idxs]
        print(out)
        used[idxs] = True
   


In [ ]:
import geopandas as gpd
from shapely.geometry import LineString
fig, ax5 = plt.subplots(figsize=(10,10))

# Convert the list of LineString objects to a GeoDataFrame
lines_gdf = gpd.GeoDataFrame(geometry=line_list, crs=perp_gdf_with_count.crs if hasattr(perp_gdf_with_count, 'crs') else "EPSG:3857")
lines_gdf = lines_gdf.to_crs(epsg=3857)
perp_gdf_with_count.to_crs(epsg=3857).plot(ax=ax5, column='count', cmap='Set1', alpha=0.5)


# Plot the lines
lines_gdf.plot(ax=ax5, color='green', linewidth=2)

plt.show()
